In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Load and prepare data
df = pd.read_csv('../data/raw/hotel_bookings.csv')

# Create new features for pricing model
def create_pricing_features(df):
    df_features = df.copy()
    
    # Check for missing values in date columns
    print("Checking for missing values in date columns:")
    print(f"arrival_date_year: {df_features['arrival_date_year'].isnull().sum()}")
    print(f"arrival_date_month: {df_features['arrival_date_month'].isnull().sum()}")
    print(f"arrival_date_day_of_month: {df_features['arrival_date_day_of_month'].isnull().sum()}")
    
    # Check the data types and sample values
    print(f"\nSample values:")
    print(f"arrival_date_year: {df_features['arrival_date_year'].head().tolist()}")
    print(f"arrival_date_month: {df_features['arrival_date_month'].head().tolist()}")
    print(f"arrival_date_day_of_month: {df_features['arrival_date_day_of_month'].head().tolist()}")
    
    # Handle missing values in date columns
    if df_features['arrival_date_year'].isnull().sum() > 0:
        df_features['arrival_date_year'] = df_features['arrival_date_year'].fillna(df_features['arrival_date_year'].mode()[0])
    
    df_features['arrival_date_month'] = df_features['arrival_date_month'].fillna('January')
    df_features['arrival_date_day_of_month'] = df_features['arrival_date_day_of_month'].fillna(1)
    
    # Convert month names to numbers
    month_mapping = {
        'January': 1, 'February': 2, 'March': 3, 'April': 4,
        'May': 5, 'June': 6, 'July': 7, 'August': 8,
        'September': 9, 'October': 10, 'November': 11, 'December': 12
    }
    
    # Map month names to numbers
    df_features['arrival_date_month_num'] = df_features['arrival_date_month'].map(month_mapping)
    
    # Handle any unmapped values (in case of typos or different formats)
    if df_features['arrival_date_month_num'].isnull().sum() > 0:
        print(f"Warning: Found unmapped month values: {df_features[df_features['arrival_date_month_num'].isnull()]['arrival_date_month'].unique()}")
        df_features['arrival_date_month_num'] = df_features['arrival_date_month_num'].fillna(1)
    
    # Convert to appropriate data types
    df_features['arrival_date_year'] = df_features['arrival_date_year'].astype(int)
    df_features['arrival_date_month_num'] = df_features['arrival_date_month_num'].astype(int)
    df_features['arrival_date_day_of_month'] = df_features['arrival_date_day_of_month'].astype(int)
    
    # Time-based features - Create datetime using numeric month
    try:
        df_features['arrival_date'] = pd.to_datetime(
            df_features[['arrival_date_year', 'arrival_date_month_num', 'arrival_date_day_of_month']].rename(
                columns={'arrival_date_month_num': 'arrival_date_month'}
            )
        )
    except Exception as e:
        print(f"Error creating datetime: {e}")
        # Alternative approach using string concatenation
        df_features['date_string'] = (
            df_features['arrival_date_year'].astype(str) + '-' +
            df_features['arrival_date_month_num'].astype(str).str.zfill(2) + '-' +
            df_features['arrival_date_day_of_month'].astype(str).str.zfill(2)
        )
        df_features['arrival_date'] = pd.to_datetime(df_features['date_string'], errors='coerce')
        df_features.drop('date_string', axis=1, inplace=True)
    
    df_features['day_of_week'] = df_features['arrival_date'].dt.dayofweek
    df_features['is_weekend'] = df_features['day_of_week'].isin([5, 6]).astype(int)
    df_features['is_peak_season'] = df_features['arrival_date_month_num'].isin([6, 7, 8, 12]).astype(int)
    
    # Handle missing values for guest counts
    df_features['children'] = df_features['children'].fillna(0)
    df_features['babies'] = df_features['babies'].fillna(0)
    
    # Booking characteristics
    df_features['total_nights'] = df_features['stays_in_weekend_nights'] + df_features['stays_in_week_nights']
    df_features['total_guests'] = df_features['adults'] + df_features['children'] + df_features['babies']
    df_features['booking_lead_time_category'] = pd.cut(
        df_features['lead_time'], 
        bins=[0, 7, 30, 90, 365], 
        labels=['Last_minute', 'Short_term', 'Medium_term', 'Long_term']
    )
    
    # Encode categorical variables
    categorical_columns = ['hotel', 'meal', 'market_segment', 'distribution_channel', 
                          'reserved_room_type', 'deposit_type', 'customer_type']
    
    for col in categorical_columns:
        if col in df_features.columns:
            le = LabelEncoder()
            df_features[f'{col}_encoded'] = le.fit_transform(df_features[col].astype(str))
    
    # Select features for modeling (use numeric month instead of original month column)
    feature_columns = [
        'lead_time', 'total_nights', 'total_guests', 'is_weekend', 'is_peak_season',
        'day_of_week', 'arrival_date_month_num', 'is_repeated_guest', 'previous_cancellations',
        'booking_changes', 'required_car_parking_spaces', 'total_of_special_requests'
    ] + [f'{col}_encoded' for col in categorical_columns if col in df_features.columns]
    
    return df_features, feature_columns

# Create features
df_processed, feature_cols = create_pricing_features(df)

# Filter out anomalies
df_processed = df_processed[
    (df_processed['adr'] > 0) & 
    (df_processed['adr'] < 1000) &
    (df_processed['is_canceled'] == 0)  # Focus on actual stays
]

print(f"Processed dataset shape: {df_processed.shape}")
print(f"Features created: {len(feature_cols)}")
print(f"Feature columns: {feature_cols}")

# Save processed data
df_processed.to_csv('../data/processed/hotel_bookings_processed.csv', index=False)
print("✅ Processed data saved!")


Checking for missing values in date columns:
arrival_date_year: 0
arrival_date_month: 0
arrival_date_day_of_month: 0

Sample values:
arrival_date_year: [2015, 2015, 2015, 2015, 2015]
arrival_date_month: ['July', 'July', 'July', 'July', 'July']
arrival_date_day_of_month: [1, 1, 1, 1, 1]
Error creating datetime: to assemble mappings requires at least that [year, month, day] be specified: [day,month,year] is missing
Processed dataset shape: (73419, 47)
Features created: 19
Feature columns: ['lead_time', 'total_nights', 'total_guests', 'is_weekend', 'is_peak_season', 'day_of_week', 'arrival_date_month_num', 'is_repeated_guest', 'previous_cancellations', 'booking_changes', 'required_car_parking_spaces', 'total_of_special_requests', 'hotel_encoded', 'meal_encoded', 'market_segment_encoded', 'distribution_channel_encoded', 'reserved_room_type_encoded', 'deposit_type_encoded', 'customer_type_encoded']
✅ Processed data saved!
